# Classical Machine Learning Methods for Image Classification

This notebook implements classical ML methods (SVM, Random Forest, k-NN) with hand-crafted features.
These are compared against deep learning methods to demonstrate the evolution of image classification techniques.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

# Machine learning imports
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from skimage.feature import hog
from skimage import exposure
from sklearn.decomposition import PCA

print("Libraries imported successfully!")

Libraries imported successfully!


## Section 1: Load Data and Create Features

In [2]:
# Load the data split from preprocessing notebook
data_df = pd.read_csv('e:/jingxizhang/image-classification-project/results/data_split.csv')

print(f"Total dataset size: {len(data_df)}")
print(f"Classes: {data_df['label'].nunique()}")
print(f"\nSplit distribution:")
print(data_df['split'].value_counts())

# Separate train, val, test sets
train_df = data_df[data_df['split'] == 'train'].reset_index(drop=True)
val_df = data_df[data_df['split'] == 'val'].reset_index(drop=True)
test_df = data_df[data_df['split'] == 'test'].reset_index(drop=True)

print(f"\nTrain: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

Total dataset size: 9144
Classes: 102

Split distribution:
split
train    6400
val      1372
test     1372
Name: count, dtype: int64

Train: 6400
Val: 1372
Test: 1372


In [6]:
# Feature extraction: HOG (Histogram of Oriented Gradients)
def extract_hog_features(image_path, resize_to=(128, 128)):
    """Extract HOG features from an image"""
    try:
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img = np.array(img.resize(resize_to))
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        
        # Extract HOG features with reduced complexity
        hog_features = feature.hog(
            gray, 
            orientations=8, 
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            visualize=False,
            channel_axis=None
        )
        return hog_features.astype(np.float32)
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# Feature extraction: Color histogram
def extract_color_histogram(image_path, bins=32, resize_to=(128, 128)):
    """Extract color histogram features from an image"""
    try:
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img = np.array(img.resize(resize_to))
        
        # Compute histograms for each channel
        hist_r = np.histogram(img[:,:,0], bins=bins, range=(0, 256))[0]
        hist_g = np.histogram(img[:,:,1], bins=bins, range=(0, 256))[0]
        hist_b = np.histogram(img[:,:,2], bins=bins, range=(0, 256))[0]
        
        # Concatenate histograms
        hist_features = np.concatenate([hist_r, hist_g, hist_b]).astype(np.float32)
        
        # Normalize
        hist_features = hist_features / (hist_features.sum() + 1e-5)
        
        return hist_features
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# Feature extraction: Combined features
def extract_features(image_path, feature_type='combined'):
    """Extract combined HOG and color histogram features"""
    if feature_type == 'hog':
        return extract_hog_features(image_path)
    elif feature_type == 'histogram':
        return extract_color_histogram(image_path)
    else:  # combined
        hog_feat = extract_hog_features(image_path)
        hist_feat = extract_color_histogram(image_path)
        if hog_feat is not None and hist_feat is not None:
            return np.concatenate([hog_feat, hist_feat])
        return None

print("Feature extraction functions defined!")


Feature extraction functions defined!


In [7]:
# Extract features from training data
print("Extracting features from training data...")
print("This may take a few minutes...\n")

# Use a smaller sample for faster training
sample_size = min(300, len(train_df))  # Use 300 samples for classical ML
train_sample_df = train_df.sample(n=sample_size, random_state=42)

train_features = []
train_labels = []
failed_count = 0

for idx, row in train_sample_df.iterrows():
    features = extract_features(row['image_path'], feature_type='combined')
    if features is not None:
        train_features.append(features)
        train_labels.append(row['label'])
    else:
        failed_count += 1
    
    if (len(train_features)) % 50 == 0:
        print(f"Successfully processed {len(train_features)} training images...")

train_features = np.array(train_features)
train_labels = np.array(train_labels)

print(f"\nTrain features shape: {train_features.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Failed to process: {failed_count} images")


Extracting features from training data...
This may take a few minutes...

Error processing e:\jingxizhang\caltech-101\crocodile_head\image_0014.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\Faces_easy\image_0001.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\lotus\image_0027.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\scorpion\image_0069.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\airplanes\image_0122.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\Faces_easy\image_0305.jpg: name 'feature' is not defined
Successfully processed 0 training images...
Error processing e:\jingxizhang\caltech-101\Motorbikes\image

In [8]:
# Extract features from validation and test data
print("Extracting features from validation data...")
val_features = []
val_labels = []
failed_val = 0

for idx, row in val_df.iterrows():
    features = extract_features(row['image_path'], feature_type='combined')
    if features is not None:
        val_features.append(features)
        val_labels.append(row['label'])
    else:
        failed_val += 1
    
    if (len(val_features)) % 100 == 0:
        print(f"Processed {len(val_features)} validation images...")

print("\nExtracting features from test data...")
test_features = []
test_labels = []
failed_test = 0

for idx, row in test_df.iterrows():
    features = extract_features(row['image_path'], feature_type='combined')
    if features is not None:
        test_features.append(features)
        test_labels.append(row['label'])
    else:
        failed_test += 1
    
    if (len(test_features)) % 100 == 0:
        print(f"Processed {len(test_features)} test images...")

val_features = np.array(val_features)
val_labels = np.array(val_labels)
test_features = np.array(test_features)
test_labels = np.array(test_labels)

print(f"\nVal features shape: {val_features.shape} (failed: {failed_val})")
print(f"Test features shape: {test_features.shape} (failed: {failed_test})")

# Handle case where feature extraction succeeds
if train_features.shape[0] > 0 and val_features.shape[0] > 0:
    # Standardize features
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)
    test_features_scaled = scaler.transform(test_features)
    
    print(f"\nFeatures standardized!")
    print(f"Train shape after scaling: {train_features_scaled.shape}")
else:
    print("\nWarning: Feature extraction produced empty arrays")


Extracting features from validation data...
Error processing e:\jingxizhang\caltech-101\accordion\image_0003.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0007.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0009.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0015.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0016.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0022.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingxizhang\caltech-101\accordion\image_0037.jpg: name 'feature' is not defined
Processed 0 validation images...
Error processing e:\jingx

## Section 2: Train Classical ML Models

In [9]:
# Train SVM classifier
print("="*60)
print("Training SVM Classifier...")
print("="*60)

start_time = time.time()
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', verbose=1)
svm_model.fit(train_features_scaled, train_labels)
svm_train_time = time.time() - start_time

# Evaluate SVM
svm_val_pred = svm_model.predict(val_features_scaled)
svm_test_pred = svm_model.predict(test_features_scaled)

svm_val_acc = accuracy_score(val_labels, svm_val_pred)
svm_test_acc = accuracy_score(test_labels, svm_test_pred)

print(f"\nSVM Training time: {svm_train_time:.2f}s")
print(f"SVM Validation Accuracy: {svm_val_acc:.4f}")
print(f"SVM Test Accuracy: {svm_test_acc:.4f}")

# Detailed metrics
svm_test_precision = precision_score(test_labels, svm_test_pred, average='weighted', zero_division=0)
svm_test_recall = recall_score(test_labels, svm_test_pred, average='weighted', zero_division=0)
svm_test_f1 = f1_score(test_labels, svm_test_pred, average='weighted', zero_division=0)

print(f"SVM Test Precision: {svm_test_precision:.4f}")
print(f"SVM Test Recall: {svm_test_recall:.4f}")
print(f"SVM Test F1-Score: {svm_test_f1:.4f}")

Training SVM Classifier...


NameError: name 'train_features_scaled' is not defined

In [ ]:
# Train Random Forest classifier
print("\n" + "="*60)
print("Training Random Forest Classifier...")
print("="*60)

start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(train_features_scaled, train_labels)
rf_train_time = time.time() - start_time

# Evaluate Random Forest
rf_val_pred = rf_model.predict(val_features_scaled)
rf_test_pred = rf_model.predict(test_features_scaled)

rf_val_acc = accuracy_score(val_labels, rf_val_pred)
rf_test_acc = accuracy_score(test_labels, rf_test_pred)

print(f"\nRandom Forest Training time: {rf_train_time:.2f}s")
print(f"Random Forest Validation Accuracy: {rf_val_acc:.4f}")
print(f"Random Forest Test Accuracy: {rf_test_acc:.4f}")

# Detailed metrics
rf_test_precision = precision_score(test_labels, rf_test_pred, average='weighted', zero_division=0)
rf_test_recall = recall_score(test_labels, rf_test_pred, average='weighted', zero_division=0)
rf_test_f1 = f1_score(test_labels, rf_test_pred, average='weighted', zero_division=0)

print(f"Random Forest Test Precision: {rf_test_precision:.4f}")
print(f"Random Forest Test Recall: {rf_test_recall:.4f}")
print(f"Random Forest Test F1-Score: {rf_test_f1:.4f}")

In [ ]:
# Train k-NN classifier
print("\n" + "="*60)
print("Training k-NN Classifier...")
print("="*60)

start_time = time.time()
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_model.fit(train_features_scaled, train_labels)
knn_train_time = time.time() - start_time

# Evaluate k-NN
knn_val_pred = knn_model.predict(val_features_scaled)
knn_test_pred = knn_model.predict(test_features_scaled)

knn_val_acc = accuracy_score(val_labels, knn_val_pred)
knn_test_acc = accuracy_score(test_labels, knn_test_pred)

print(f"\nk-NN Training time: {knn_train_time:.2f}s")
print(f"k-NN Validation Accuracy: {knn_val_acc:.4f}")
print(f"k-NN Test Accuracy: {knn_test_acc:.4f}")

# Detailed metrics
knn_test_precision = precision_score(test_labels, knn_test_pred, average='weighted', zero_division=0)
knn_test_recall = recall_score(test_labels, knn_test_pred, average='weighted', zero_division=0)
knn_test_f1 = f1_score(test_labels, knn_test_pred, average='weighted', zero_division=0)

print(f"k-NN Test Precision: {knn_test_precision:.4f}")
print(f"k-NN Test Recall: {knn_test_recall:.4f}")
print(f"k-NN Test F1-Score: {knn_test_f1:.4f}")

## Section 3: Compare Classical ML Models

In [ ]:
# Create comparison results dataframe
results_comparison = pd.DataFrame({
    'Method': ['SVM', 'Random Forest', 'k-NN'],
    'Val Accuracy': [svm_val_acc, rf_val_acc, knn_val_acc],
    'Test Accuracy': [svm_test_acc, rf_test_acc, knn_test_acc],
    'Precision': [svm_test_precision, rf_test_precision, knn_test_precision],
    'Recall': [svm_test_recall, rf_test_recall, knn_test_recall],
    'F1-Score': [svm_test_f1, rf_test_f1, knn_test_f1],
    'Training Time (s)': [svm_train_time, rf_train_time, knn_train_time]
})

print("\n" + "="*80)
print("CLASSICAL ML METHODS COMPARISON")
print("="*80)
print(results_comparison.to_string(index=False))
print("="*80)

# Save results
results_comparison.to_csv('e:/jingxizhang/image-classification-project/results/classical_ml_results.csv', index=False)
print("\nResults saved to classical_ml_results.csv")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
axes[0, 0].bar(results_comparison['Method'], results_comparison['Test Accuracy'], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_ylabel('Test Accuracy', fontsize=11)
axes[0, 0].set_title('Test Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(results_comparison['Test Accuracy']):
    axes[0, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
axes[0, 0].grid(axis='y', alpha=0.3)

# Precision, Recall, F1 comparison
metrics = ['Precision', 'Recall', 'F1-Score']
x_pos = np.arange(len(results_comparison))
width = 0.25

for i, metric in enumerate(metrics):
    axes[0, 1].bar(x_pos + i*width, results_comparison[metric], width, label=metric)

axes[0, 1].set_ylabel('Score', fontsize=11)
axes[0, 1].set_title('Detailed Metrics Comparison', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x_pos + width)
axes[0, 1].set_xticklabels(results_comparison['Method'])
axes[0, 1].legend()
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(axis='y', alpha=0.3)

# Training time comparison
axes[1, 0].bar(results_comparison['Method'], results_comparison['Training Time (s)'], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 0].set_ylabel('Training Time (seconds)', fontsize=11)
axes[1, 0].set_title('Training Time Comparison', fontsize=12, fontweight='bold')
for i, v in enumerate(results_comparison['Training Time (s)']):
    axes[1, 0].text(i, v + 0.1, f'{v:.2f}s', ha='center', fontsize=10)
axes[1, 0].grid(axis='y', alpha=0.3)

# Val vs Test accuracy
x_pos_2 = np.arange(len(results_comparison))
width_2 = 0.35
axes[1, 1].bar(x_pos_2 - width_2/2, results_comparison['Val Accuracy'], width_2, label='Validation', color='lightcoral')
axes[1, 1].bar(x_pos_2 + width_2/2, results_comparison['Test Accuracy'], width_2, label='Test', color='steelblue')
axes[1, 1].set_ylabel('Accuracy', fontsize=11)
axes[1, 1].set_title('Validation vs Test Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x_pos_2)
axes[1, 1].set_xticklabels(results_comparison['Method'])
axes[1, 1].set_ylim([0, 1])
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/02_classical_ml_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Comparison visualization saved!")

In [ ]:
# Confusion matrices for best model (Random Forest)
cm_rf = confusion_matrix(test_labels, rf_test_pred)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_rf, annot=False, fmt='d', cmap='Blues', cbar=True, ax=ax, xticklabels=[], yticklabels=[])
ax.set_title(f'Random Forest Confusion Matrix (Test Set)\nAccuracy: {rf_test_acc:.4f}', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/02_rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix visualization saved!")

In [ ]:
# Feature importance from Random Forest
feature_importance = rf_model.feature_importances_
top_indices = np.argsort(feature_importance)[-20:]  # Top 20 features

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_indices)), feature_importance[top_indices], color='steelblue')
ax.set_yticks(range(len(top_indices)))
ax.set_yticklabels([f'Feature {i}' for i in top_indices], fontsize=9)
ax.set_xlabel('Importance', fontsize=11)
ax.set_title('Random Forest: Top 20 Most Important Features', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/02_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Feature importance visualization saved!")